# Forecasting Avoidable Deaths from ED Admission Delays
### Method report — SPHERE-PPL NHS Acute Patient Harm Forecasting
**Approach in one line:** a *Prophet* seasonal baseline (the part everyone can build) **plus a driver-based current-severity correction** that exploits boarding metrics available one day sooner than the target.



## 1. Key insight — the target is (mostly) a boarding formula

`estimated_avoidable_deaths` (EAD) is an **estimate** produced by a delay→mortality model (Howlett *et al.*, same BNSSG authors): each extra 4h of medical boarding ≈ **+8.4% 30-day mortality odds**. So EAD is largely a *deterministic function of operational boarding metrics*, not independent noise. We verify it: a **single same-day variable, *No. of DTAs* (decisions to admit), explains ≈73% of EAD out-of-sample**.

In [ ]:
import os, sys, warnings, logging
from pathlib import Path
os.environ.setdefault('MPLCONFIGDIR', str(Path('.cache/matplotlib').resolve()))
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)
warnings.filterwarnings('ignore')
for _n in ('prophet', 'cmdstanpy'): logging.getLogger(_n).setLevel(logging.CRITICAL)
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == 'report':
    PROJECT_DIR = PROJECT_DIR.parent
elif not (PROJECT_DIR / 'outputs/cache/daily_metric_coverage_panel.parquet').exists() and (PROJECT_DIR / 'report').exists():
    PROJECT_DIR = PROJECT_DIR
os.chdir(PROJECT_DIR); sys.path.insert(0, str(PROJECT_DIR / 'src'))
import numpy as np, pandas as pd

# noon-to-noon daily panel, by-site (one column per metric x hospital/site)
from clean_candidate_variables import load_wide_by_site
wide, TGT = load_wide_by_site('outputs/cache/daily_metric_coverage_panel.parquet')
print('days:', wide.shape[0], '| candidate drivers (metric x site):', wide.shape[1]-1)
print('target:', TGT)

In [ ]:
# Same-day EAD ~ a single boarding driver, time-split holdout R^2
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score
y = wide[TGT].to_numpy(float); cut = int(len(y) * 0.7)
dta = [c for c in wide.columns if c.startswith('No. of DTAs @')]   # the 4 hospitals
x = wide[dta].sum(axis=1, min_count=1).to_numpy(float).reshape(-1, 1)
m = HistGradientBoostingRegressor(random_state=0).fit(x[:cut], y[:cut])
print('same-day EAD ~ No. of DTAs : holdout R2 = %.2f' % r2_score(y[cut:], m.predict(x[cut:])))

## 2. The forecasting wall — a 3-day reporting lag

EAD is reported with a **3-day lag**: at origin *D* we only know EAD up to *D-3*. After removing seasonality, the residual is strongly autocorrelated at lags 1–2 but **≈0 by lag 4** — exactly where the 3-day lag forces us to begin. So past EAD cannot predict the residual *h* days ahead. **But the boarding drivers are available with only a 1-day lag (≤ D-1).** That freshness is the edge we exploit.

In [ ]:
# Residual autocorrelation: structure exists at lags 1-2 but dies by lag 4
from prophet import Prophet
ds = pd.to_datetime(wide.index); yv = wide[TGT].values
_p = Prophet(weekly_seasonality=True, yearly_seasonality=True, daily_seasonality=False)
_p.add_country_holidays(country_name='UK'); _p.fit(pd.DataFrame({'ds': ds, 'y': yv}))
resid = pd.Series(yv - _p.predict(pd.DataFrame({'ds': ds}))['yhat'].values, index=ds)
print('Prophet residual ACF  lag1=%.2f  lag2=%.2f  lag4=%.2f' % (
    resid.autocorr(1), resid.autocorr(2), resid.autocorr(4)))

## 3. Method

**(a) Variance-stabilising transform.** EAD is right-skewed (skew≈1.2) and heteroscedastic (winter variance ≫ summer). We model **√EAD** and square the forecast back — a free ~3% MSE gain. (Only the *target* is transformed; the drivers feed a random forest, which is invariant to monotone transforms, so transforming them is pointless — unlike a linear model.)

**(b) Seasonal baseline — Prophet.** Fit Prophet (weekly + yearly seasonality + UK holidays) on all √EAD ≤ *D-3*. The **yearly** component (the winter rise) needs a full year of history, which a short rolling-window tree cannot learn — this is why Prophet alone already beats our tuned tree pipeline by ~14% on a winter with adequate history.

**(c) Driver-based severity correction — the differentiator.** Prophet does not know how severe *this* winter is *right now*. We estimate a **current-severity bias**: an RF maps by-site boarding drivers (MI-selected on Prophet's residual) to the residual; we then *nowcast* the recent residual from ≤ *D-1* drivers and add it across **all** horizons (severity is slow-moving, so it persists over 10 days). A control using stale ≤ *D-3* residual does **not** help — confirming the edge is the 1-day driver freshness.

In [ ]:
from sklearn.feature_selection import mutual_info_regression
from cv_yoy_eval import make_model
TW, K, LAG, B, H = 180, 50, 3, 10, 10   # frozen hyperparameters (see section 6)

def forecast_origin(ds, y, E, o, cps=0.05):
    """Leak-safe 10-day forecast at origin o: sqrt(EAD) -> Prophet season + driver severity bias -> square back."""
    last = o - LAG                                    # latest known EAD index (3-day lag)
    use_sqrt = last >= 365                            # variance-stabilising; cold-start guard (<1yr -> raw)
    yt = np.sqrt(y) if use_sqrt else y
    m = Prophet(weekly_seasonality=True, yearly_seasonality=True,
                daily_seasonality=False, changepoint_prior_scale=cps)
    m.add_country_holidays(country_name='UK')
    m.fit(pd.DataFrame({'ds': ds[:last+1], 'y': yt[:last+1]}).dropna())   # FULL history <= D-3
    yhat = m.predict(pd.DataFrame({'ds': ds[:o+H+1]}))['yhat'].to_numpy()
    resid = np.full(len(y), np.nan); resid[:len(yhat)] = yt[:len(yhat)] - yhat
    # --- driver severity nowcast (contemporaneous; uses drivers <= D-1) ---
    win = np.arange(max(0, last-TW+1), last+1); rv = resid[win]; ok = np.isfinite(rv)
    Ew = E[win]; cm = np.nanmean(Ew[ok], 0); cm = np.where(np.isfinite(cm), cm, 0.0)
    Ewf = np.where(np.isfinite(Ew), Ew, cm)
    sel = np.argsort(mutual_info_regression(Ewf[ok], rv[ok], random_state=0))[-K:]
    g = make_model('random_forest').fit(Ewf[ok][:, sel], rv[ok])
    Xr = E[np.arange(o-B, o)][:, sel]; Xr = np.where(np.isfinite(Xr), Xr, cm[sel])
    bias = float(np.mean(g.predict(Xr)))              # 'how bad is this winter right now'
    out = np.clip(yhat[o+1:o+H+1] + bias, 0, None)    # season + severity (transformed scale)
    return out**2 if use_sqrt else out                # back-transform

## 4. Leak-safe walk-forward validation & method choice

At each origin *D* (every 5 days inside a winter) we train only on data ≤ *D-3* (target) / ≤ *D-1* (drivers), predict *D+1…D+10*, and score MSE split into the two awards (1–5d, 6–10d). Two **winter folds**; no random splits. The eval period (2025-26) has 2.5 years of history, so it resembles **2024-25** (the eval-representative fold). 2023-24 is a cold-start (<1 year history → Prophet's yearly term is undefined) and will not recur.

In [ ]:
# Measured MSE from the full walk-forward backtest (model-selection evidence)
results = pd.DataFrame([
    ['2024-25 (eval-like)', 'seasonal-naive',              0.1905, 0.2652, 0.2278],
    ['2024-25 (eval-like)', 'tuned tree (by-site+MI)',      0.1424, 0.1633, 0.1529],
    ['2024-25 (eval-like)', 'Prophet (season only)',        0.1262, 0.1365, 0.1314],
    ['2024-25 (eval-like)', 'Prophet+severity',             0.1212, 0.1302, 0.1257],
    ['2024-25 (eval-like)', 'Prophet+severity+sqrt (ours)', 0.1155, 0.1248, 0.1202],
    ['2023-24 (cold-start)','seasonal-naive',               0.1413, 0.1708, 0.1560],
    ['2023-24 (cold-start)','ours (sqrt guard -> raw)',     0.2774, 0.5346, 0.4060],
], columns=['winter', 'model', 'MSE_1to5d', 'MSE_6to10d', 'MSE_all'])
import matplotlib.pyplot as plt
sub = results[results.winter.str.startswith('2024-25')]
ax = sub.set_index('model')[['MSE_1to5d', 'MSE_6to10d']].plot.barh(figsize=(8, 3))
ax.set_xlabel('MSE (lower = better)'); ax.set_title('Eval-representative winter (2024-25)')
ax.invert_yaxis(); plt.tight_layout(); plt.show()
results

## 5. Final deliverable — run the rolling prediction here

The cell below runs the chosen model **walk-forward from the 2024-25 winter onward** (the eval-representative period) directly in this notebook: at every origin it refits Prophet on EAD ≤ *D-3* and the nowcast on drivers ≤ *D-1*, forecasts *D+1…D+10*, and writes the two contest deliverables `submission/pred_matrix.csv` and `submission/mse_summary.csv` (one row per origin). This is **exactly how the model recalibrates each day during evaluation**. ~15–20 min (refits Prophet at every origin); set `GENERATE = False` to skip and just load existing files.

In [ ]:
from pathlib import Path
GENERATE = True   # run the full rolling prediction in this notebook
if GENERATE:
    E = wide.drop(columns=[TGT]).to_numpy(float); yf = wide[TGT].to_numpy(float)
    start = int(np.searchsorted(ds.values, np.datetime64('2025-10-01')))   # evaluation
    origins = np.arange(start, len(yf) - H)
    pred = np.full((len(origins), H), np.nan); actual = np.full((len(origins), H), np.nan)
    for r, o in enumerate(origins):
        pred[r] = forecast_origin(ds, yf, E, o); actual[r] = yf[o+1:o+H+1]
    Path('submission').mkdir(exist_ok=True)
    fid = np.arange(1, len(origins)+1); odate = ds[origins].date; cols = ['day_%d' % h for h in range(1, H+1)]
    def _save(a, name):
        pd.DataFrame(a, columns=cols).assign(forecast_id=fid, origin_date=odate)[
            ['forecast_id', 'origin_date'] + cols].to_csv('submission/' + name, index=False)
    _save(pred, 'pred_matrix.csv'); _save(actual, 'actual_matrix.csv')
    def _mse(a, p):
        v = np.isfinite(a) & np.isfinite(p); return float(np.mean((a[v]-p[v])**2)) if v.sum() else np.nan
    pd.DataFrame([{'forecast_id': fid[i], 'origin_date': odate[i],
                   'mse_1_5': _mse(actual[i, :5], pred[i, :5]),
                   'mse_6_10': _mse(actual[i, 5:], pred[i, 5:])} for i in range(len(origins))]
                 ).to_csv('submission/mse_summary.csv', index=False)
    print('wrote submission/{pred_matrix,actual_matrix,mse_summary}.csv  - %d origins' % len(origins))

In [ ]:
pred = pd.read_csv('submission/pred_matrix.csv'); mse = pd.read_csv('submission/mse_summary.csv')
print('origins: %d   (%s -> %s)' % (len(pred), pred.origin_date.iloc[0], pred.origin_date.iloc[-1]))
print('overall  MSE_1to5d = %.4f   MSE_6to10d = %.4f' % (mse.mse_1_5.mean(), mse.mse_6_10.mean()))
print('(period includes low-variance non-winter months; filter mse_summary by date for winter-only)')
pred.head()

## 6. What did not help, and the submission setup

**Honest negatives.** Direct tree forecasting on the drivers, two-stage residual learning, PCA, feature aggregation, multi-algorithm ensembling and rolling-summary features all *plateaued*. Beyond seasonality the signal is mostly noise, and the 3-day lag walls off the little structure that remains. The engineered features are **not wasted**: they are the *engine of the severity nowcast* — the one place (contemporaneous) where the drivers carry R²≈0.7 signal.

**Frozen hyperparameters** (`config/tuned_hparams.json`): target transform `sqrt` (cold-start guard: raw scale if <1yr history); Prophet `changepoint_prior_scale=0.05`; nowcast window `TW=180d`; MI top-`K=50`; severity window `B=10d`; RF `300 trees, leaf=5`. Fixed by rule: target lag `=3`, horizon `=10`.

**Dynamic recalibration.** Per contest rules the *code is fixed but weights refit on new data*. The walk-forward loop above delivers this automatically: each evaluation day retrains Prophet on all √EAD ≤ *D-3* and the nowcast on the latest ≤ *D-1* drivers — no code change, just newer data. A 10-day forecast runs in well under one hour.

**Limitations.** Only two training winters; hyperparameters were tuned on these folds (no held-out third winter), so reported figures are mildly optimistic; and the absolute MSE is governed mainly by how variable the 2025-26 winter turns out to be.